# AI Web Search v2 — Agent Loop with Tool Use

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiejun-ai/ai_websearch/blob/main/v2/ai_websearch.ipynb)

**Input:** User Query  
**Architecture:** Agent Loop with Tool Use and Multiple LLM Calls

1. Define a `web_search` function as a **tool** the LLM can invoke via the Tavily API.
2. Run an **Agent Loop** (up to `MAX_LOOP_TIMES` LLM calls):
   - The LLM decides when and what to search.
   - Each search result is labeled `[n]`; results accumulate and are deduplicated across iterations.
   - Loop exits when the LLM returns an answer instead of a tool call.
3. Render the output as HTML in this notebook:
   - **Answer Section** — LLM answer with inline citations `[n](URL)`
   - **Web Search Results Section** — all unique results in discovery order

In [ ]:
import os, json
import litellm
from tavily import TavilyClient
from IPython.display import display, HTML
import markdown

# --- API Keys ----------------------------------------------------------------
TAVILY_API_KEY = "tvly-..."  # your Tavily API key
OPENAI_API_KEY = "sk-..."    # your OpenAI API key
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# --- Configuration -----------------------------------------------------------
MODEL          = "gpt-4o-mini"  # change to e.g. "gpt-4o", "claude-3-5-haiku-20241022"
TOP_K          = 10             # max web results per search
MAX_LOOP_TIMES = 3              # max LLM calls per request (including final answer)

litellm.set_verbose = False

## Step 1: Define Web Search as a Tool

`search_web` fetches results from Tavily. `WEB_SEARCH_TOOL` is the function schema exposed to the LLM so it can call this tool when needed.

In [ ]:
def search_web(query, k=TOP_K):
    """Return top-k Tavily results for query."""
    response = TavilyClient(api_key=TAVILY_API_KEY).search(query, max_results=k)
    return response["results"]  # each: {title, url, content, score}


WEB_SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Search the web for current information on a topic.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query"}
            },
            "required": ["query"]
        }
    }
}

## Step 2: Agent Loop

The loop initializes the conversation with the user query, then repeatedly calls the LLM:
- If the LLM issues a `web_search` tool call, we execute the search, number the new results globally (`[n]`), deduplicate by URL, and add the results to the message context.
- If the LLM returns a plain text response (no tool call), we exit and return it as the answer.

The loop runs at most `MAX_LOOP_TIMES` times (= max total LLM calls).

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful research assistant with access to a web_search tool. "
    "Use it to find information needed to answer the user's query; you may search multiple times with different queries. "
    "Each search result is labeled [n] with its URL. "
    "Cite supporting text in your answer as [n](URL) using those numbers. "
    "When you have enough information, write a clear, comprehensive markdown answer."
)


def run_agent(query, model=MODEL, max_loops=MAX_LOOP_TIMES):
    """Run the agent loop; return (answer_text, all_results)."""
    seen_urls = {}   # url -> result dict (augmented with "num" key)
    counter   = 0    # global discovery counter across all searches
    messages  = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": query},
    ]

    last_content = ""
    for _ in range(max_loops):
        response = litellm.completion(
            model=model, messages=messages,
            tools=[WEB_SEARCH_TOOL], tool_choice="auto"
        )
        msg = response.choices[0].message
        last_content = msg.content or ""

        if not msg.tool_calls:  # LLM produced a plain answer — exit loop
            return last_content, list(seen_urls.values())

        # --- Execute tool calls and add results to context ---
        messages.append(msg)
        for tc in msg.tool_calls:
            search_query = json.loads(tc.function.arguments)["query"]
            results      = search_web(search_query)

            tool_text = ""
            for r in results:
                if r["url"] not in seen_urls:  # deduplicate; first occurrence wins
                    counter += 1
                    r["num"] = counter
                    seen_urls[r["url"]] = r
                num = seen_urls[r["url"]]["num"]
                tool_text += f"[{num}] {r['title']}\nURL: {r['url']}\n{r['content']}\n\n"

            messages.append({
                "role": "tool", "tool_call_id": tc.id,
                "name": "web_search", "content": tool_text
            })

    # Max loops reached — return whatever the LLM last said
    return last_content, list(seen_urls.values())

## Step 3: Format and Display Output

`format_output` assembles the final markdown:
- **Answer** section: the LLM's response (with `[n](URL)` citations).
- **Web Search Results** section: all unique results in discovery order (`[1]`, `[2]`, …).

`display_answer` converts markdown to HTML and renders it in the notebook.

In [ ]:
def format_output(answer_text, all_results):
    """Combine LLM answer and search results into a single markdown string."""
    # Display results in ascending [n] (discovery) order
    ordered = sorted(all_results, key=lambda r: r["num"])

    results_md = "## Web Search Results\n\n"
    for r in ordered:
        results_md += f"### [{r['num']}] [{r['title']}]({r['url']})\n{r['content']}\n\n"

    return f"## Answer\n\n{answer_text}\n\n---\n\n{results_md}"


def display_answer(md_text):
    """Render markdown as HTML in the notebook."""
    html = markdown.markdown(md_text, extensions=["extra"])
    display(HTML(html))

## Run the Demo

Set your query below and run the cell. The agent will search the web as needed and display a cited answer.

In [ ]:
query = "What are the latest AI breakthroughs in 2025?"  # <- change this

answer_text, all_results = run_agent(query)
output_md = format_output(answer_text, all_results)
display_answer(output_md)